#    Libraries

In [ ]:
import cv2
import os
import time
from datetime import datetime
from ultralytics import YOLO 

#    Environment Setup

In [ ]:
os.makedirs("Intruders", exist_ok=True) # Create folder for images if it doesn't exist
print("Loading the model...")
model = YOLO("yolov8n.pt") # Load the smallest and fastest version of YOLOv8
print("Model loaded successfully! Starting the camera... (Press 'Q' to exit)")

#    Camera Initialization

In [ ]:
cap = cv2.VideoCapture(0) # 0 refers to the primary laptop camera

last_save_time = 0
cooldown_seconds = 5 # The model will wait 5 seconds before capturing the same person again

#    Real-time Analysis Loop & Drawing & Decision Making (All in ONE Cell)

In [ ]:
# Real-time Analysis Loop & Drawing & Decision Making (All in ONE Cell)
while True:
    # 1. Read Camera
    ret, frame = cap.read()
    if not ret:
        print("Error: Could not read from the camera!")
        break

    # 2. AI Detection
    results = model(frame, classes=[0], conf=0.5, verbose=False) 
    person_detected = False

    # 3. Draw Boxes
    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, "Intruder!", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            person_detected = True

    # 4. Decision Making
    current_time = time.time()
    if person_detected and (current_time - last_save_time) > cooldown_seconds:
        now = datetime.now()
        time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
        
        # Save the image
        image_path = f"Intruders/intruder_{time_str}.jpg"
        cv2.imwrite(image_path, frame)
        
        # Log the event
        with open("log.txt", "a", encoding="utf-8") as log_file:
            log_file.write(f"⚠ Intruder detected in frame at: {now.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
        print(f"⚠ Alert: Image of the intruder captured and saved as {image_path}")
        last_save_time = current_time

    # 5. Display the live video feed
    cv2.imshow("Smart Security Camera", frame)

    # 6. Exit when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up memory and release the camera
cap.release()
cv2.destroyAllWindows()